# MAZ: Multiplayer AlphaZero on Kaggle GPU

Train a 3-player Connect Four agent using AlphaZero self-play.

**Setup:** Go to **Settings (right panel) → Accelerator → GPU P100** before running.

Kaggle gives 30 hours/week of GPU. This notebook is tuned for a ~2 hour session
with 128 games/gen and 50 MCTS sims.

In [ ]:
# Install JAX for CUDA and MAZ
!pip install -q jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q git+https://github.com/leungchristopher/maz.git#egg=maz[colab]

In [ ]:
# Verify GPU is available and JAX is using it
import jax
devices = jax.devices()
print(f'JAX devices ({len(devices)}): {devices}')
print(f'Default backend: {jax.default_backend()}')
assert jax.default_backend() == 'gpu', f'JAX is using {jax.default_backend()}, not GPU! Check CUDA install.'

# Quick smoke test — ensure computation runs on GPU
x = jax.numpy.ones(1)
print(f'Array device: {x.devices()}')
print('GPU verified!')

In [ ]:
import os

# Kaggle persists /kaggle/working/ when you save the notebook
CHECKPOINT_DIR = '/kaggle/working/maz_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Persistent JIT compilation cache — survives kernel restarts
JAX_CACHE_DIR = '/kaggle/working/jax_cache'
os.makedirs(JAX_CACHE_DIR, exist_ok=True)
os.environ['JAX_CACHE_DIR'] = JAX_CACHE_DIR

print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'JAX cache dir: {JAX_CACHE_DIR}')

In [ ]:
# Override default config for GPU-saturating training
# T4 handles batch=128 as fast as batch=1, so use large batches
import maz.main as main_module

main_module.NUM_GAMES_PER_GEN = 128   # batch=128 for net.apply
main_module.NUM_SIMULATIONS = 50      # deep search
main_module.NUM_GENERATIONS = 100     # ~2 hrs on T4
main_module.BATCH_SIZE = 128          # match game batch for training
main_module.EPOCHS_PER_WINDOW = 4     # more training per generation
main_module.PEAK_LR = 2e-3           # higher LR for larger batches

print(f'Config: {main_module.NUM_GAMES_PER_GEN} games/gen, '
      f'{main_module.NUM_SIMULATIONS} sims/move, '
      f'{main_module.NUM_GENERATIONS} generations')

In [ ]:
# Run training (resumes automatically from last checkpoint)
from maz.main import main

main(
    checkpoint_dir=CHECKPOINT_DIR,
    use_wandb=False,
    resume=True,
)